# 05 — Train + calibrate the consistency gate (the actual novelty)

Partner B, weeks 2-3. Trained ONLY on legitimate training-split features. Threshold
calibrated on legitimate VALIDATION-split features only — never on test, never on attacks.

In [ ]:
!python -m src.consistency.train_consistency --config configs/default.yaml \
    --features results/features/features.csv \
    --out results/consistency

## Ablation baselines to train alongside the gate

1. `single_feature_gate` — repeat gate training once per feature in isolation.
2. `features_as_classifier_input` — an ordinary supervised classifier on the features.

In [ ]:
import pandas as pd
from src.consistency.gate import ConsistencyGate
from src.consistency.calibrate import select_threshold_for_fpr
from src.features.pipeline import FEATURE_COLUMNS
from src.utils.config import load_config

config = load_config('configs/default.yaml')
df = pd.read_csv('results/features/features.csv')
train_df, val_df = df[df['split'] == 'train'], df[df['split'] == 'val']
target_fpr = config['fusion']['target_val_false_positive_rate']

single_feature_gates = {}
for col in FEATURE_COLUMNS:
    gate = ConsistencyGate(method=config['consistency_gate']['method'], mode=config['consistency_gate']['mode'])
    gate.fit(train_df[[col]].values, train_df['tx_id'].values)
    scores = gate.score(val_df[[col]].values, val_df['tx_id'].values)
    threshold = select_threshold_for_fpr(scores, target_fpr)
    single_feature_gates[col] = (gate, threshold)
    print(col, 'threshold =', threshold)

In [ ]:
from src.evaluation.combine import FeaturesAsInputClassifier, calibrate_confidence_threshold

features_clf = FeaturesAsInputClassifier().fit(train_df[FEATURE_COLUMNS].values, train_df['tx_id'].values)
_, val_conf = features_clf.predict_and_confidence(val_df[FEATURE_COLUMNS].values)
features_clf_threshold = calibrate_confidence_threshold(val_conf, target_fpr)
print('features-as-input classifier threshold =', features_clf_threshold)